# Preparación de datos: FaceForensics++ (C23) desde Kaggle

Descarga el dataset [`xdxd003/ff-c23`](https://www.kaggle.com/datasets/xdxd003/ff-c23), extrae los rostros, crea los splits y guarda un `.zip` en Drive que luego usa `01_entrenamiento_colab.ipynb`.

**Credenciales de Kaggle:** en kaggle.com → *Settings* → *API* → *Generate New Token*. Guarda el token en los **Secretos** de Colab (icono de llave) con el nombre `KAGGLE_API_TOKEN`. `kagglehub` lo lee automáticamente.

> **Alternativa recomendada: el Mac.** Colab tiene solo 2 CPU y la extracción depende de la CPU. En el Mac (10 núcleos) es varias veces más rápida y no se desconecta. Los comandos son los mismos (ver README). Esta extracción sí se puede reanudar: si se corta, vuelve a correr la celda.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/deepfakes'
os.makedirs(DRIVE, exist_ok=True)

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [ ]:
REPO = 'Julianchu850/proyecto-redes-neuronales-deteccion-deepfake'
try:
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
except Exception:
    url = f'https://github.com/{REPO}.git'
if not os.path.exists('/content/repo'):
    !git clone -q {url} /content/repo
%cd /content/repo
!pip install -q -U -r requirements.txt kagglehub

## 1. Descargar el dataset

In [ ]:
import kagglehub
RAIZ_FFPP = kagglehub.dataset_download('xdxd003/ff-c23')
print(RAIZ_FFPP)
!du -sh {RAIZ_FFPP}
!find {RAIZ_FFPP} -maxdepth 2 -type d | head -20

## 2. Extraer rostros

32 frames por video real y 8 por falso, para que las clases queden balanceadas. Si el dataset trae FaceShifter, también se extrae, pero solo se usa como técnica nunca vista (E3b).

In [ ]:
!python -m src.data.extraer_rostros --dataset ffpp --raiz {RAIZ_FFPP} --salida /content/data/faces/ffpp \
    --procesos {os.cpu_count()}

## 3. Splits sin fugas de identidad

In [ ]:
!python -m src.data.crear_splits --dataset ffpp --rostros /content/data/faces/ffpp --salida data/splits/ffpp
!mkdir -p {DRIVE}/splits && cp -r data/splits/ffpp {DRIVE}/splits/

## 4. Guardar en Drive

Se empaqueta sin recomprimir (`-0`): los PNG ya están comprimidos y así es mucho más rápido.

In [ ]:
%cd /content/data/faces
!zip -q -r -0 /content/faces_ffpp.zip ffpp
!cp /content/faces_ffpp.zip {DRIVE}/faces_ffpp.zip
!ls -lh {DRIVE}/faces_ffpp.zip
%cd /content/repo

## 5. Vista rápida de los rostros extraídos

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
m = pd.read_csv('/content/data/faces/ffpp/manifiesto.csv', dtype={'video_id': str})
print(m.groupby('metodo').agg(rostros=('ruta', 'size'), videos=('video', 'nunique')))
muestra = m.groupby('metodo').sample(4, random_state=0)
fig, ax = plt.subplots(len(muestra) // 4, 4, figsize=(9, 2.3 * len(muestra) // 4))
for a, (_, f) in zip(ax.flat, muestra.iterrows()):
    a.imshow(Image.open(f'/content/data/faces/ffpp/{f.ruta}')); a.set_title(f.metodo, fontsize=8); a.axis('off')
plt.tight_layout()